# Aula 16 · Hands-on: construindo cargas de trabalho dinâmicas com tarefas avançadas

## O que é
A implementação, no job `growdev_pipeline_emprestimos`, do DAG desenhado na aula 15.

## Por que importa
É o que torna um job capaz de **reagir aos próprios dados**, em vez de seguir sempre o mesmo caminho fixo.

## Como aplicar
### 1. For each na bronze
1. Abra o job → clique na tarefa `bronze` → apague-a (vamos recriá-la).
2. **+ Add task → For each**.
   - **Task name:** `bronze` · **Depends on:** `preparar_ambiente`
   - **Inputs:** `["customers", "accounts", "account_balances", "loans"]`
   - **Concurrency:** `2` (duas tabelas por vez)
3. Dentro do For each, **Add a task to loop over**:
   - **Task name:** `bronze_tabela` · **Type:** Notebook · **Path:** `Projeto/01_bronze`
   - **Parameters:** `tabela` = `{{input}}`
4. Na tarefa `silver`, confira se **Depends on** = `bronze`.

> `{{input}}` é substituído por cada item da lista: o mesmo notebook roda 4 vezes, uma para cada tabela.

### 2. Tarefa que publica um task value
1. **+ Add task → Notebook** → `validar_qualidade` · Path `Projeto/04_validar_qualidade` · Depends on `silver`.
2. Abra o notebook e encontre a linha `dbutils.jobs.taskValues.set(key="qualidade_ok", ...)`: é o "recado".

### 3. If/else condition
1. **+ Add task → If/else condition** → `qualidade_ok` · Depends on `validar_qualidade`.
2. Condição: `{{tasks.validar_qualidade.values.qualidade_ok}}` **==** `true`.
3. Edite a tarefa `gold`: **Depends on** = `qualidade_ok (true)`.
4. **+ Add task → Notebook** → `registrar_alerta_qualidade` · Path `Projeto/05_registrar_alerta_qualidade` · Depends on `qualidade_ok (false)` · Parameters `job_run_id` = `{{job.run_id}}`.

### 4. Run if
1. **+ Add task → Notebook** → `registrar_execucao` · Path `Projeto/07_registrar_execucao`.
2. **Depends on:** `gold` **e** `registrar_alerta_qualidade`.
3. **Run if dependencies:** **All done**.
4. **Parameters:** `job_run_id` = `{{job.run_id}}`.

### 5. Parâmetro do limite de qualidade
**Job parameters** → adicione `limite_nulos_pct` = `5`.

### 6. Testando os dois caminhos
- **Caminho verdadeiro:** **Run now**. A validação aprova (cerca de 3% de nulos, abaixo de 5%) e a `gold` roda. `registrar_alerta_qualidade` aparece como **Excluded**.
- **Caminho falso:** **Run now with different parameters** → `limite_nulos_pct` = `0`. A validação reprova, a `gold` fica **Excluded** e o alerta é registrado.
- Nos dois casos, `registrar_execucao` roda (Run if: All done).

Para ver o "recado" de uma tarefa: abra a execução → clique em `validar_qualidade` → **Task values**.

In [ ]:
%sql
SELECT 'execucao' AS tipo, job_run_id, qualidade_ok AS detalhe, registrado_em FROM growdev.gold.log_execucoes
UNION ALL
SELECT 'alerta_qualidade', job_run_id, motivo, registrado_em FROM growdev.gold.log_alertas_qualidade
ORDER BY registrado_em DESC

## Checklist da aula
- [ ] A bronze virou um For each sobre as 4 tabelas
- [ ] `validar_qualidade` publica o task value `qualidade_ok`
- [ ] O If/else desvia para `gold` ou para `registrar_alerta_qualidade`
- [ ] `registrar_execucao` roda sempre (Run if: All done)
- [ ] Testei os dois caminhos e sei ler o DAG de cada um